# Figure 10
This notebook reproduce Fig. 10 in [Ronchi et al. 2021](https://ui.adsabs.harvard.edu/abs/2021ApJ...916..100R/abstract).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import utilities.plot_settings

In [ ]:
RA_galcen = 266.4  # Galactic center position in ra [deg]
DEC_galcen = -29.0  # Galactic center position in dec [deg]
DEG_TO_MAS = 3600000  # Convert [deg] to [mas]

In [ ]:
# Load the ATNF catalog.
data_atnf = pd.read_csv("../../data/observations/atnf_full_nobinary_13-11-2020.csv", delimiter=";", header=[0,1])
data_atnf.head()

In [ ]:
# Remove the rows that contains NAN in the columns we want to consider 
data_atnf = data_atnf[~data_atnf["RAJD"]["(deg)"].isin(['NAN'])]
data_atnf = data_atnf[~data_atnf["DECJD"]["(deg)"].isin(['NAN'])]

RA_atnf = data_atnf["RAJD"]["(deg)"].to_numpy().astype(np.float64) 
DEC_atnf = data_atnf["DECJD"]["(deg)"].to_numpy().astype(np.float64) 

In [ ]:
# Read observed proper motion neutron stars .csv file.
data_pm = pd.read_csv("../../data/observations/PSRs_prop_motion_22-05-2020.csv", header=[0,1])
data_pm.head()

In [ ]:
# Select only stars with measure of P and Pdot and that are not in globular clusters.
data_pm = data_pm[~data_pm["P0"]["[s]"].isin(['NAN'])]
data_pm = data_pm[~data_pm["P1"]["[s/s]"].isin(['NAN'])]
data_pm = data_pm[~data_pm["DIST_DM"]["[kpc]"].isin(['NAN'])]
data_pm = data_pm[~data_pm["ASSOC"]["Unnamed: 24_level_1"].isin(['EXGAL:SMC', 'EXGAL:LMC', 'GC:47Tuc', 'GC:M3', 'GC:M5', 'GC:M13', 'GC:NGC6440', 'GC:Ter5', 'GC:NGC6441', 'GC:NGC6517', 'GC:NGC6522', 'GC:NGC6624', 'GC:M28(NGC6626)', 'GC:NGC6652', 'GC:M22(NGC6656)', 'GC:NGC6752', 'GC:NGC6760', 'GC:M15', 'GC:M30'])]

In [ ]:
# Extract parameters.
RA_pm = data_pm["RAJD"]["[deg]"].to_numpy().astype(np.float64) 
DEC_pm = data_pm["DECJD"]["[deg]"].to_numpy().astype(np.float64) 
pmRA_pm = data_pm["PMRA"]["[mas/yr]"].to_numpy().astype(np.float64) 
pmDEC_pm = data_pm["PMDEC"]["[mas/yr]"].to_numpy().astype(np.float64) 
dist_pm = data_pm["DIST_DM"]["[kpc]"].to_numpy().astype(np.float64) 
NS_class_pm = data_pm["CLASS"]["Unnamed: 13_level_1"].to_numpy()
P_pm = data_pm["P0"]["[s]"].to_numpy().astype(np.float64) 
Pdot_pm = data_pm["P1"]["[s/s]"].to_numpy().astype(np.float64) 
assoc_pm = data_pm["ASSOC"]["Unnamed: 24_level_1"].to_numpy()

# select only isolated non recycled neutron stars i.e. with Pdot > 1e-17
cond = (Pdot_pm > 1e-17) & (NS_class_pm != "Binary PSR") & (dist_pm < 20)
RA_pm = RA_pm[cond]
DEC_pm = DEC_pm[cond]
pmRA_pm = pmRA_pm[cond]
pmDEC_pm = pmDEC_pm[cond]
dist_pm = dist_pm[cond]

In [ ]:
# plot NS in the ICRS frame
fig, ax = plt.subplots(figsize=(17,9))

ax.set_xlim(0., 360.)
ax.set_ylim(-90., 90.)
ax.set_xlabel('RA [deg]')
ax.set_ylabel('DEC [deg]')

ax.scatter(
    RA_atnf,
    DEC_atnf,
    linestyle="None",
    marker="o",
    facecolors="tab:gray",
    edgecolors="none",
    s=20,
    alpha=0.2,
    rasterized=True,
    zorder=0
)

scatter = ax.scatter(
    RA_pm,
    DEC_pm,
    linestyle="None",
    marker="o",
    c=dist_pm,
    cmap = "jet",
    s=40,
    alpha=1.,
    rasterized=True,
)
cbar = fig.colorbar(scatter, ax = ax)
cbar.set_label(r'$d_{\odot}$ [kpc]')

# plot the trajectories of the observed neutron stars in the equatorial frame on a timescale of 0.5 Myr
t = np.linspace(0., 5.e5, 100)

for i in range(len(RA_pm)):

    RA_traj = RA_pm[i] - pmRA_pm[i] * t / DEG_TO_MAS
    DEC_traj = DEC_pm[i] - pmDEC_pm[i] * t / DEG_TO_MAS
    ax.plot(
        RA_traj,
        DEC_traj,
        linestyle="-",
        linewidth=1.5,
        color="black",
        alpha=1,
        rasterized=True,
    )
ax.plot(RA_galcen, DEC_galcen, marker="*", markeredgewidth=1, color="tab:red", markersize=30, zorder=0)

plt.savefig(
    f"plots/Figure10.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()